In [ ]:
%cd ../..

: 

In [ ]:
from hedging.simulators import HestonSimulator

: 

In [2]:
from hedging.envs import HedgeCallHeston, HedgeDocHeston
from experiments.utils.ppo_mlp_actor import create_ppo_mlp_actor
from experiments.utils.training_loop import action_training
from experiments.utils.testing import test_model
from experiments.utils.sim_config import (
    load_heston_data,
    EnvConfig,
    PPOConfig,
    TrainingConfig,
    compute_barriers
)
import wandb

import numpy as np
import torch
"""
if not hasattr(torch.Tensor, "_orig_numpy"):
    torch.Tensor._orig_numpy = torch.Tensor.numpy

    def safe_numpy(self, *args, **kwargs):
        # transparently move tensor to CPU before numpy() if it's on GPU
        if self.is_cuda:
            return self.detach().cpu().numpy()
        return self._orig_numpy(*args, **kwargs)

    torch.Tensor.numpy = safe_numpy
"""
from torchrl.envs import GymWrapper
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

# --- Env Parameters ---
params, S0, K, v0 = load_heston_data()
env_cfg = EnvConfig()
ppo_cfg = PPOConfig()
train_cfg = TrainingConfig()

In [3]:
r = env_cfg.r
maturity = env_cfg.maturity
trap = env_cfg.trap
num_paths = 1
num_steps = 250
history_len = env_cfg.history_len
input_dim = 17
hidden_size = train_cfg.hidden_size
action_dim = 1
transaction_cost = env_cfg.transaction_cost
transaction_fee_rate = env_cfg.transaction_fee_rate


In [4]:
base_env = HedgeCallHeston(
    S0=S0, K = K, r=r, v0=v0, theta=params["theta"], rho=params["rho"],
    kappa=params["kappa"], xi=params["sigma"], maturity=maturity,
    num_steps=num_steps, num_paths=num_paths, history_len=history_len, 
    transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
)

env = GymWrapper(base_env)

In [5]:
H = compute_barriers(K)
base_env = HedgeDocHeston(
    S0=S0, K = K, H=H, r=r, v0=v0, theta=params["theta"], rho=params["rho"],
    kappa=params["kappa"], xi=params["sigma"], maturity=maturity,
    num_steps=num_steps, num_paths=num_paths, history_len=history_len,
    transaction_cost=transaction_cost, transaction_fee_rate=transaction_fee_rate
)
env = GymWrapper(base_env)

In [6]:
tensor_dict = env.reset()

HestonSimulator initialized on TFRT_CPU_0


In [7]:
env.call_prices.shape

(1, 5, 5, 251)

In [10]:
tensor_dict["observation"].shape


torch.Size([25, 1, 17])